# 7 · RFP Parser
## Long-Context Extraction
⏱ ~20 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/7-rfp-parser/rfp_parser_workbook.ipynb)

Reads a procurement RFP and extracts a typed `RFPExtraction` — deadlines, requirements, scoring criteria — from dense document text.

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The extraction problem |
| 2 | Nested schema design |
| 3 | Optional fields |
| 4 | Parse a realistic RFP |
| 5 | Post-processing in pure Python |
| star | Exercise + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - Why Structured Extraction?

Free text from an LLM:
```
"Budget around $750K, deadline in July, FedRAMP cert required..."
```

Structured extraction:
```python
result.budget_ceiling       # '$750,000'
result.deadlines[0].date    # '2026-07-15'
result.requirements[0].mandatory  # True
```

One schema. One LLM call. All requirements extracted, typed, and sortable.

---

## Part 2 - Nested Schema

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class Deadline(BaseModel):
    label: str
    date: str = Field(description="YYYY-MM-DD if parseable, else raw text")
    is_hard: bool


class Requirement(BaseModel):
    id: str = Field(description="e.g. REQ-01")
    category: Literal["technical", "administrative", "legal", "financial"]
    text: str
    mandatory: bool


class ScoringCriterion(BaseModel):
    criterion: str
    weight_percent: Optional[int] = Field(default=None, ge=0, le=100)
    description: str


class RFPExtraction(BaseModel):
    title: str
    issuing_agency: str
    budget_ceiling: Optional[str] = None
    contract_duration: Optional[str] = None
    deadlines: List[Deadline]
    requirements: List[Requirement]
    scoring_criteria: List[ScoringCriterion]
    summary: str

print("Schema ready.")

---

## Part 3 - Optional Fields

`Optional[str] = None` tells the model to return null if the information is absent.

Key prompt instruction: **If a field is not mentioned, use null or an empty list.**

This prevents hallucination of missing data.

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = SystemMessage(
    "You are a procurement analyst extracting structured data from RFPs.\n\n"
    "Extract: title, issuing_agency, budget_ceiling (null if absent), "
    "contract_duration (null if absent), "
    "deadlines (label, date YYYY-MM-DD, is_hard), "
    "requirements (id REQ-01..., category, text, mandatory), "
    "scoring_criteria (criterion, weight_percent, description), "
    "and summary (two sentences).\n\n"
    "If a field is not mentioned: use null or empty list.\n"
    "Extract ALL requirements."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = SYSTEM_PROMPT | llm.with_structured_output(RFPExtraction)
print("Parser ready.")

---

## Part 4 - Parse a Realistic RFP

In [ ]:
SAMPLE_RFP = (
    "REQUEST FOR PROPOSALS - City of Riverside IT Division\n"
    "RFP IT-2026-041: Enterprise Cloud Migration and Managed Services\n"
    "ISSUING AGENCY: City of Riverside, Office of Information Technology\n"
    "CONTRACT DURATION: 3 years with two 1-year renewal options\n"
    "BUDGET: Not to exceed $750,000 for the initial three-year term.\n\n"
    "IMPORTANT DATES:\n"
    "- Pre-proposal Conference (mandatory): 2026-06-20\n"
    "- Questions Due: 2026-07-01\n"
    "- Proposal Submission Deadline: 2026-07-15 (HARD DEADLINE)\n"
    "- Award Notification: 2026-08-01 (estimated)\n\n"
    "MANDATORY REQUIREMENTS:\n"
    "REQ-01 (Technical): Vendor must be a certified FedRAMP-authorized cloud provider partner.\n"
    "REQ-02 (Legal): Vendor must be registered to do business in California.\n"
    "REQ-03 (Administrative): Vendor must carry minimum $2M general liability insurance.\n"
    "REQ-04 (Technical): All data must remain within US jurisdiction.\n"
    "REQ-05 (Financial): Vendor must provide audited financials for the past two years.\n\n"
    "PREFERRED QUALIFICATIONS:\n"
    "REQ-06 (Technical): Municipal IT modernization experience preferred.\n"
    "REQ-07 (Technical): ISO 27001 certification preferred.\n\n"
    "EVALUATION CRITERIA:\n"
    "1. Technical Approach - 35%\n"
    "2. Relevant Experience - 25%\n"
    "3. Cost and Value - 30%\n"
    "4. Staff Qualifications - 10%\n"
)

result = parser.invoke(SAMPLE_RFP)
print(f"Title:    {result.title}")
print(f"Agency:   {result.issuing_agency}")
print(f"Budget:   {result.budget_ceiling}")
print(f"Duration: {result.contract_duration}")
print(f"Summary:  {result.summary}")
print(f"\n{len(result.deadlines)} deadlines, {len(result.requirements)} requirements, {len(result.scoring_criteria)} criteria")

In [ ]:
print("DEADLINES")
for d in result.deadlines:
    flag = " [HARD]" if d.is_hard else ""
    print(f"  {d.date:12}  {d.label}{flag}")

print("\nREQUIREMENTS")
for r in result.requirements:
    flag = "[mandatory]" if r.mandatory else "[preferred]"
    print(f"  {r.id}  {flag}  {r.text[:65]}")

print("\nSCORING")
for c in sorted(result.scoring_criteria, key=lambda x: x.weight_percent or 0, reverse=True):
    w = f"{c.weight_percent}%" if c.weight_percent is not None else "N/A"
    print(f"  {w:5}  {c.criterion}")

---

## Part 5 - Post-Processing in Pure Python

One LLM call extracts. Then pure Python analyzes - no further LLM calls.

In [ ]:
mandatory = [r for r in result.requirements if r.mandatory]
preferred = [r for r in result.requirements if not r.mandatory]
hard = [d for d in result.deadlines if d.is_hard]
top = max(result.scoring_criteria, key=lambda c: c.weight_percent or 0)

print(f"Mandatory requirements: {len(mandatory)}")
print(f"Preferred qualifications: {len(preferred)}")
print(f"Hard deadlines: {len(hard)}")
if hard:
    print(f"  Next hard deadline: {hard[0].date}")
print(f"\nFocus proposal on: {top.criterion} ({top.weight_percent}%)")

---

## Exercise - Parse Without a Budget

Write a short RFP (8 sentences) with NO budget or duration mentioned.
Run the parser and confirm `budget_ceiling` and `contract_duration` are both `None`.

This tests the Optional field behavior - the model must not invent a value when the information is absent.

In [ ]:
# Exercise Starter
NO_BUDGET_RFP = (
    "RFP Title: Website Redesign Services\n"
    "Issuing Agency: Riverside Public Library System\n\n"
    "The Library seeks proposals to redesign our public website.\n"
    "The new site must be mobile-responsive and ADA-compliant.\n"
    "Vendor must have experience with at least three public-sector clients.\n"
    "Proposals are due by August 15, 2026. This is a hard deadline.\n"
    "Evaluation: design quality (50%) and relevant experience (50%).\n"
)

# TODO: parse NO_BUDGET_RFP and verify budget_ceiling is None

In [ ]:
# Exercise Answer Key
result2 = parser.invoke(NO_BUDGET_RFP)
print(f"budget_ceiling:    {result2.budget_ceiling}")    # None
print(f"contract_duration: {result2.contract_duration}") # None
print(f"deadlines:         {len(result2.deadlines)}")
print(f"requirements:      {len(result2.requirements)}")
print(f"scoring_criteria:  {len(result2.scoring_criteria)}")

---

## What's Next?

The same pattern works for contracts, legal briefs, earnings calls, medical records - any document where structured data is buried in prose.

**Next:** Example 8 - Multi-Agent Research (supervisor coordinates specialist sub-agents)

---
*Example 7 of 8 - agent-use-cases*